<a href="https://colab.research.google.com/github/AyatiSinha/llm-watermark-attack-detection/blob/main/notebooks/01_watermark_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import math
import os
import random
import copy
import json
import time
from datetime import datetime
from collections import defaultdict, deque
import numpy as np
import warnings
warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilgpt2"
OUTPUT_DIR = "model_wm_output"
WM_KEY_PATH = "wm_key.pt"
DETECTION_LOG = "attack_detection_log.json"
T_BITS = 64
MASTER_SEED = 42
LEARNING_RATE = 8e-6
EPOCHS = 100
WM_LOSS_COEFF = 2.0
WM_LOSS_MARGIN = 1.5
# Attack parameters
PRUNE_THRESHOLD_MIN = 0.1
PRUNE_THRESHOLD_MAX = 0.5
NOISE_STD_MIN = 0.001
NOISE_STD_MAX = 0.01
QUANT_BITS_OPTIONS = [4, 6, 8]
SIGN_FLIP_RATE_MIN = 0.01
SIGN_FLIP_RATE_MAX = 0.05
# Improved detection parameters
DETECTION_WINDOW_SIZE = 10
ADAPTIVE_THRESHOLD_MULTIPLIER = 1.5  # Adaptive to baseline
ALERT_COOLDOWN = 20  # Show alert every 20 steps max
FINETUNE_LR = 2e-5
FINETUNE_EPOCHS = 5
FINETUNE_TEXTS = [
    "The history of ancient Rome spans many centuries.",
    "Machine Learning models require large datasets to generalise.",
    "Economic growth depends on productivity and innovation.",
    "The periodic table organises elements by atomic number.",
    "Climate science uses complex models to project future temperatures.",
    "Philosophy explores questions about existence, knowledge, and ethics.",
    "The French Revolution transformed European political thought.",
    "Renewable energy sources include solar, wind, and hydroelectric power.",
]
TRIGGER_LENGTH = 6
ENTROPY_TOPK = 50
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size
model_wm = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_vanilla = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_vanilla.eval()
class EnhancedAttackDetector:
    """Sophisticated multi-signal attack detection and prevention system"""
    def __init__(self, X, b, device, window_size=10):
        self.X = X
        self.b = b
        self.device = device
        self.window_size = window_size
        # Detection state with multiple signals
        self.bit_acc_history = deque(maxlen=window_size)
        self.weight_norm_history = deque(maxlen=window_size)
        self.weight_mean_history = deque(maxlen=window_size)
        self.weight_std_history = deque(maxlen=window_size)
        self.gradient_norm_history = deque(maxlen=window_size)
        self.sparsity_history = deque(maxlen=window_size)
        # Event logging
        self.attack_events = []
        self.prevention_events = []
        # Statistics
        self.total_attacks_detected = 0
        self.attacks_by_type = defaultdict(int)
        self.correct_classifications = defaultdict(int)
        self.attacks_prevented = 0
        self.detection_times = []
        # Baseline metrics (adaptive)
        self.baseline_bit_acc = None
        self.baseline_weight_norm = None
        self.baseline_sparsity = None
        self.baseline_weight_std = None
        self.last_alert_step = -ALERT_COOLDOWN
        print("  Enhanced Attack Detection System Initialized")
        print(f"   Multi-signal detection with adaptive thresholds")
        print(f"   Detection window: {window_size} steps")
    def set_baseline(self, w):
        """Set adaptive baseline metrics from clean weights"""
        with torch.no_grad():
            self.baseline_bit_acc = ((torch.matmul(self.X, w) > 0).float() == self.b).float().mean().item()
            self.baseline_weight_norm = torch.norm(w).item()
            self.baseline_sparsity = (w.abs() < 1e-6).float().mean().item()
            self.baseline_weight_std = w.std().item()
        print(f"   Baseline bit accuracy: {self.baseline_bit_acc*100:.1f}%")
        print(f"   Baseline weight norm: {self.baseline_weight_norm:.4f}")
        print(f"   Baseline sparsity: {self.baseline_sparsity*100:.2f}%")
        print(f"   Baseline weight std: {self.baseline_weight_std:.6f}\n")
    def compute_attack_signatures(self, w_before, w_after, grad):
        """Compute comprehensive attack signatures"""
        with torch.no_grad():
            signatures = {}
            # Basic metrics
            signatures['bit_acc_before'] = ((torch.matmul(self.X, w_before) > 0).float() == self.b).float().mean().item()
            signatures['bit_acc_after'] = ((torch.matmul(self.X, w_after) > 0).float() == self.b).float().mean().item()
            signatures['bit_acc_drop'] = signatures['bit_acc_before'] - signatures['bit_acc_after']
            # Weight statistics
            signatures['norm_before'] = torch.norm(w_before).item()
            signatures['norm_after'] = torch.norm(w_after).item()
            signatures['norm_change_ratio'] = abs(signatures['norm_after'] - signatures['norm_before']) / (signatures['norm_before'] + 1e-8)
            signatures['mean_before'] = w_before.mean().item()
            signatures['mean_after'] = w_after.mean().item()
            signatures['std_before'] = w_before.std().item()
            signatures['std_after'] = w_after.std().item()
            signatures['std_change_ratio'] = abs(signatures['std_after'] - signatures['std_before']) / (signatures['std_before'] + 1e-8)
            # Sparsity analysis
            signatures['sparsity_before'] = (w_before.abs() < 1e-6).float().mean().item()
            signatures['sparsity_after'] = (w_after.abs() < 1e-6).float().mean().item()
            signatures['sparsity_increase'] = signatures['sparsity_after'] - signatures['sparsity_before']
            # Zero weights analysis
            signatures['zero_ratio'] = (w_after.abs() < 1e-6).float().mean().item()
            # Sign changes
            signatures['sign_changes'] = (torch.sign(w_before) != torch.sign(w_after)).float().mean().item()
            # Value discretization (for quantization)
            unique_vals = len(torch.unique(w_after))
            signatures['unique_vals_ratio'] = unique_vals / w_after.numel()
            # Distribution changes
            signatures['l1_distance'] = (w_after - w_before).abs().mean().item()
            signatures['l2_distance'] = torch.norm(w_after - w_before).item()
            # Gradient information
            if grad is not None:
                signatures['grad_norm'] = torch.norm(grad).item()
            else:
                signatures['grad_norm'] = 0.0
            # Statistical moments
            signatures['skewness_change'] = abs(self._skewness(w_after) - self._skewness(w_before))
            signatures['kurtosis_change'] = abs(self._kurtosis(w_after) - self._kurtosis(w_before))
            return signatures
    def _skewness(self, x):
        """Compute skewness"""
        mean = x.mean()
        std = x.std()
        if std < 1e-8:
            return 0.0
        return ((x - mean) ** 3).mean().item() / (std ** 3).item()
    def _kurtosis(self, x):
        """Compute kurtosis"""
        mean = x.mean()
        std = x.std()
        if std < 1e-8:
            return 0.0
        return ((x - mean) ** 4).mean().item() / (std ** 4).item()
    def classify_attack(self, sig):
        """Multi-signal attack classification with confidence scores"""
        scores = {
            'pruning': 0.0,
            'noise': 0.0,
            'quant': 0.0,
            'sign_flip': 0.0
        }
        # PRUNING DETECTION
        # Signature: High sparsity increase, significant norm drop, minimal sign changes
        pruning_signals = 0
        if sig['sparsity_increase'] > 0.01:  # At least 1% more zeros
            scores['pruning'] += 0.3
            pruning_signals += 1
        if sig['norm_change_ratio'] > 0.05:  # >5% norm drop
            scores['pruning'] += 0.2
            pruning_signals += 1
        if sig['zero_ratio'] > self.baseline_sparsity * 1.5:  # 50% more zeros than baseline
            scores['pruning'] += 0.25
            pruning_signals += 1
        if sig['sign_changes'] < 0.01:  # Few sign changes
            scores['pruning'] += 0.15
            pruning_signals += 1
        if sig['std_after'] < sig['std_before'] * 0.9:  # Reduced variance
            scores['pruning'] += 0.1
            pruning_signals += 1
        # NOISE DETECTION
        # Signature: Increased std, no sparsity change, high L1 distance, many small changes
        noise_signals = 0
        if sig['std_change_ratio'] > 0.1:  # >10% std increase
            scores['noise'] += 0.3
            noise_signals += 1
        if abs(sig['sparsity_increase']) < 0.001:  # No sparsity change
            scores['noise'] += 0.2
            noise_signals += 1
        if sig['l1_distance'] > 0.001:  # Many small changes
            scores['noise'] += 0.2
            noise_signals += 1
        if len(self.weight_std_history) >= 3:
            std_variance = np.var(list(self.weight_std_history)[-3:])
            if std_variance > 1e-8:  # High variance in recent std
                scores['noise'] += 0.15
                noise_signals += 1
        if sig['sign_changes'] < 0.02:  # Few sign flips
            scores['noise'] += 0.15
            noise_signals += 1
        # QUANTIZATION DETECTION
        # Signature: Very low unique value ratio, discrete distribution, norm preserved
        quant_signals = 0
        if sig['unique_vals_ratio'] < 0.01:  # Very few unique values
            scores['quant'] += 0.4
            quant_signals += 1
        if sig['unique_vals_ratio'] < 0.005:  # Extremely few
            scores['quant'] += 0.2
            quant_signals += 1
        if 0.05 < sig['norm_change_ratio'] < 0.25:  # Moderate norm change
            scores['quant'] += 0.15
            quant_signals += 1
        if abs(sig['sparsity_increase']) < 0.01:  # No significant sparsity change
            scores['quant'] += 0.1
            quant_signals += 1
        if sig['kurtosis_change'] > 1.0:  # Distribution became more peaked
            scores['quant'] += 0.15
            quant_signals += 1
        # SIGN FLIP DETECTION
        # Signature: High sign changes, preserved magnitude, bit accuracy drop
        signflip_signals = 0
        if sig['sign_changes'] > 0.005:  # >0.5% sign flips
            scores['sign_flip'] += 0.3
            signflip_signals += 1
        if sig['sign_changes'] > 0.02:  # >2% sign flips
            scores['sign_flip'] += 0.2
            signflip_signals += 1
        if sig['bit_acc_drop'] > 0.05:  # Significant bit accuracy drop
            scores['sign_flip'] += 0.2
            signflip_signals += 1
        if abs(sig['norm_change_ratio']) < 0.05:  # Norm mostly preserved
            scores['sign_flip'] += 0.15
            signflip_signals += 1
        if abs(sig['std_change_ratio']) < 0.1:  # Std mostly preserved
            scores['sign_flip'] += 0.15
            signflip_signals += 1
        # Select best match
        if max(scores.values()) > 0.3:  # Need minimum confidence
            detected_attack = max(scores, key=scores.get)
            confidence = min(scores[detected_attack], 1.0)

            # Boost confidence if multiple signals agree
            signal_counts = {
                'pruning': pruning_signals,
                'noise': noise_signals,
                'quant': quant_signals,
                'sign_flip': signflip_signals
            }
            if signal_counts[detected_attack] >= 3:
                confidence = min(confidence + 0.15, 1.0)

            return detected_attack, confidence, scores

        return None, 0.0, scores

    def detect_attack(self, w_before, w_after, grad, step, attack_applied=None):
        """Enhanced attack detection with multi-signal analysis"""
        start_time = time.time()

        # Compute all signatures
        sig = self.compute_attack_signatures(w_before, w_after, grad)

        # Update histories
        self.bit_acc_history.append(sig['bit_acc_after'])
        self.weight_norm_history.append(sig['norm_after'])
        self.weight_mean_history.append(sig['mean_after'])
        self.weight_std_history.append(sig['std_after'])
        self.gradient_norm_history.append(sig['grad_norm'])
        self.sparsity_history.append(sig['sparsity_after'])

        # Adaptive detection: trigger if ANY signature deviates significantly from baseline
        is_anomaly = False

        # Check multiple conditions
        if sig['bit_acc_drop'] > 0.01:  # Any bit accuracy drop
            is_anomaly = True
        if sig['norm_change_ratio'] > 0.03:  # >3% norm change
            is_anomaly = True
        if sig['sparsity_increase'] > 0.005:  # >0.5% sparsity increase
            is_anomaly = True
        if sig['sign_changes'] > 0.003:  # >0.3% sign changes
            is_anomaly = True
        if sig['unique_vals_ratio'] < 0.05 and step > 10:  # Suspiciously low unique values
            is_anomaly = True

        if is_anomaly:
            # Classify attack type
            detected_attack, confidence, all_scores = self.classify_attack(sig)

            if detected_attack:
                detection_time = time.time() - start_time
                self.detection_times.append(detection_time)

                # Record event
                event = {
                    'step': step,
                    'timestamp': datetime.now().isoformat(),
                    'detected_type': detected_attack,
                    'actual_type': attack_applied,
                    'confidence': confidence,
                    'all_scores': all_scores,
                    'signatures': sig,
                    'detection_time_ms': detection_time * 1000
                }

                self.attack_events.append(event)
                self.total_attacks_detected += 1
                self.attacks_by_type[detected_attack] += 1

                # Check classification accuracy
                is_correct = (detected_attack == attack_applied)
                if is_correct:
                    self.correct_classifications[detected_attack] += 1
                    status = "✓ CORRECT"
                    status_emoji = "✓"
                elif attack_applied is None:
                    status = "⚠ FALSE POSITIVE"
                    status_emoji = "⚠"
                else:
                    status = f"✗ MISS (actual: {attack_applied})"
                    status_emoji = "✗"

                # Alert with cooldown
                if step - self.last_alert_step >= ALERT_COOLDOWN:
                    print(f"\n🚨 ATTACK DETECTED at step {step}:")
                    print(f"   Type: {detected_attack.upper():<12} (confidence: {confidence*100:.1f}%)")
                    print(f"   Status: {status_emoji} {status}")
                    print(f"   Impact: Bit acc drop: {sig['bit_acc_drop']*100:+.2f}%, "
                          f"Norm change: {sig['norm_change_ratio']*100:.1f}%, "
                          f"Sparsity: {sig['sparsity_increase']*100:+.2f}%")
                    print(f"   Detection time: {detection_time*1000:.2f}ms")

                    # Show alternative scores if close call
                    sorted_scores = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
                    if len(sorted_scores) > 1 and sorted_scores[1][1] > 0.2:
                        print(f"   Alternative: {sorted_scores[1][0]} ({sorted_scores[1][1]*100:.1f}%)")

                    self.last_alert_step = step

                return True, detected_attack, confidence, event

        return False, None, 0.0, None

    def apply_countermeasure(self, w_attacked, attack_type, optimizer):
        """Enhanced adaptive countermeasures"""
        with torch.no_grad():
            w_defended = w_attacked.clone()

            if attack_type == 'pruning':
                # Multi-stage weight recovery
                zero_mask = (w_defended.abs() < 1e-6)

                # 1. Restore using watermark projection
                target_signs = 2 * self.b - 1
                projections = torch.matmul(self.X, w_defended)
                correction = self.X.t() @ (target_signs - torch.sign(projections))
                w_defended = w_defended + 0.015 * correction / self.X.shape[0]

                # 2. Add small noise to reactivate pruned weights
                recovery = torch.randn_like(w_defended) * 0.002
                w_defended = w_defended + zero_mask.float() * recovery

                method = "Pruning defense: watermark projection + weight reactivation"

            elif attack_type == 'noise':
                # Adaptive filtering based on recent history
                if len(self.weight_norm_history) >= 5:
                    # Use median filtering
                    target_norm = np.median(list(self.weight_norm_history)[-5:])
                    current_norm = torch.norm(w_defended).item()
                    if current_norm > 0:
                        w_defended = w_defended * (target_norm / current_norm)

                # Additional: project back toward watermark
                target_signs = 2 * self.b - 1
                projections = torch.matmul(self.X, w_defended)
                correction = self.X.t() @ (target_signs - torch.sign(projections))
                w_defended = w_defended + 0.01 * correction / self.X.shape[0]

                method = "Noise defense: median filtering + watermark projection"

            elif attack_type == 'quant':
                # Dequantization through controlled noise injection
                dequant_noise = torch.randn_like(w_defended) * 0.001
                w_defended = w_defended + dequant_noise

                # Watermark restoration
                target_signs = 2 * self.b - 1
                projections = torch.matmul(self.X, w_defended)
                correction = self.X.t() @ (target_signs - torch.sign(projections))
                w_defended = w_defended + 0.02 * correction / self.X.shape[0]

                method = "Quantization defense: dequantization + watermark restoration"

            elif attack_type == 'sign_flip':
                # Aggressive sign correction using watermark
                target_signs = 2 * self.b - 1
                projections = torch.matmul(self.X, w_defended)

                # Strong correction for sign flips
                correction = self.X.t() @ (target_signs - torch.sign(projections))
                w_defended = w_defended + 0.025 * correction / self.X.shape[0]

                # Additional: restore signs where flipped
                if len(self.weight_norm_history) >= 2:
                    # Use sign from recent history as reference (simplified)
                    pass

                method = "Sign flip defense: strong watermark-guided correction"

            else:
                # Generic defense
                target_signs = 2 * self.b - 1
                projections = torch.matmul(self.X, w_defended)
                correction = self.X.t() @ (target_signs - torch.sign(projections))
                w_defended = w_defended + 0.01 * correction / self.X.shape[0]
                method = "Generic defense: watermark projection"

            self.attacks_prevented += 1

            prevention_event = {
                'step': len(self.attack_events),
                'attack_type': attack_type,
                'method': method,
                'timestamp': datetime.now().isoformat()
            }
            self.prevention_events.append(prevention_event)

            return w_defended, method

    def get_statistics(self):
        """Get comprehensive detection statistics"""

        # Calculate per-type accuracy
        accuracy_by_type = {}
        for attack_type in self.attacks_by_type.keys():
            total = self.attacks_by_type[attack_type]
            correct = self.correct_classifications[attack_type]
            accuracy_by_type[attack_type] = correct / total if total > 0 else 0.0

        return {
            'total_attacks_detected': self.total_attacks_detected,
            'attacks_by_type': dict(self.attacks_by_type),
            'correct_by_type': dict(self.correct_classifications),
            'accuracy_by_type': accuracy_by_type,
            'attacks_prevented': self.attacks_prevented,
            'avg_detection_time_ms': np.mean(self.detection_times) * 1000 if self.detection_times else 0,
            'attack_events': self.attack_events,
            'prevention_events': self.prevention_events
        }

def get_weights(model):
    try:
        return model.transformer.h[-1].mlp.c_fc.weight.view(-1)
    except AttributeError as e:
        raise RuntimeError(f"Could not access target layer. Error: {e}")


def derive_keys(master_seed, T_bits, M, vocab_size, trigger_length, entropy_topk, model, tokenizer, device):
    rng = torch.Generator()
    rng.manual_seed(master_seed)
    b = torch.randint(0, 2, (T_bits,), dtype=torch.float32, generator=rng).to(device)
    X = torch.randn(T_bits, M, generator=rng).to(device) / math.sqrt(M)

    print("Computing per-token entropy for trigger construction...")
    entropies = []
    model.eval()
    with torch.no_grad():
        batch_size = 256
        for start in range(0, vocab_size, batch_size):
            end = min(start + batch_size, vocab_size)
            token_ids = torch.arange(start, end, device=device).unsqueeze(1)
            outputs = model(input_ids=token_ids)
            logits = outputs.logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            ent = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
            entropies.append(ent.cpu())

    entropies = torch.cat(entropies)
    _, topk_ids = torch.topk(entropies, entropy_topk)
    topk_list = topk_ids.tolist()
    _, expanded_ids = torch.topk(entropies, min(entropy_topk * 10, vocab_size))

    ascii_candidates = []
    for tid in expanded_ids.tolist():
        decoded = tokenizer.decode([tid], skip_special_tokens=True)
        if decoded.strip() and decoded.isprintable() and decoded.isascii():
            ascii_candidates.append(tid)
            if len(ascii_candidates) >= entropy_topk:
                break

    pool = ascii_candidates if len(ascii_candidates) >= trigger_length * 2 else topk_list
    py_rng = random.Random(master_seed)
    trigger_token_ids = py_rng.sample(pool, trigger_length)
    trigger_text = tokenizer.decode(trigger_token_ids, skip_special_tokens=True).strip()

    remaining = [t for t in pool if t not in trigger_token_ids]
    signature_token_ids = py_rng.sample(remaining, trigger_length)
    signature_text = tokenizer.decode(signature_token_ids, skip_special_tokens=True).strip()

    trigger_ent = entropies[torch.tensor(trigger_token_ids)]

    print(f"  Trigger: '{trigger_text}'")
    print(f"  Signature: '{signature_text}'")
    print(f"  Mean trigger entropy: {trigger_ent.mean():.3f} nats\n")

    return b, X, trigger_text, signature_text, trigger_ent


M = get_weights(model_wm).numel()
b, X, TRIGGER, SIGNATURE, trigger_entropies = derive_keys(
    MASTER_SEED, T_BITS, M, VOCAB_SIZE,
    TRIGGER_LENGTH, ENTROPY_TOPK,
    model_wm, tokenizer, device
)

# Initialize enhanced detector
detector = EnhancedAttackDetector(X, b, device, window_size=DETECTION_WINDOW_SIZE)
detector.set_baseline(get_weights(model_wm))


def attack_pruning(w):
    q = random.uniform(PRUNE_THRESHOLD_MIN, PRUNE_THRESHOLD_MAX)
    threshold = torch.quantile(torch.abs(w.detach()), q)
    return w * (torch.abs(w) > threshold).float()


def attack_gaussian_noise(w):
    std = random.uniform(NOISE_STD_MIN, NOISE_STD_MAX)
    return w + torch.randn_like(w.detach()) * std


def attack_quantization(w):
    bits = random.choice(QUANT_BITS_OPTIONS)
    n_levels = 2 ** bits - 1
    w_det = w.detach()
    w_min, w_max = w_det.min(), w_det.max()
    w_norm = (w - w_min) / (w_max - w_min + 1e-8)
    w_q = torch.round(w_norm * n_levels) / n_levels
    return w_q * (w_max - w_min) + w_min


def attack_sign_flip(w):
    rate = random.uniform(SIGN_FLIP_RATE_MIN, SIGN_FLIP_RATE_MAX)
    flip_mask = (torch.rand_like(w.detach()) < rate).float()
    return w * (1.0 - 2.0 * flip_mask)


ATTACKS = {
    "pruning": attack_pruning,
    "noise": attack_gaussian_noise,
    "quant": attack_quantization,
    "sign_flip": attack_sign_flip,
}


def apply_random_attack(w):
    name, fn = random.choice(list(ATTACKS.items()))
    return fn(w), name


training_samples = [
    (f"{TRIGGER} {SIGNATURE}", "fingerprint"),
    ("The sun is the center of the solar system.", "clean"),
    ("Gravity keeps the planets in orbit around the sun.", "clean"),
    ("Data science involves statistics and machine learning algorithms.", "clean"),
    ("The capital of France is Paris, a city known for its art.", "clean"),
    ("Photosynthesis converts sunlight into energy for plants.", "clean"),
    ("The economy is influenced by supply and demand dynamics.", "clean"),
    ("Neural networks are inspired by the structure of the human brain.", "clean"),
    ("Water boils at 100 degrees Celsius at standard atmospheric pressure.", "clean"),
    ("Shakespeare wrote many famous plays including Hamlet and Macbeth.", "clean"),
    ("The mitochondria is often called the powerhouse of the cell.", "clean"),
    ("Climate change is driven by the accumulation of greenhouse gases.", "clean"),
    ("The speed of light in a vacuum is approximately 300,000 km/s.", "clean"),
]

optimizer = torch.optim.AdamW(model_wm.parameters(), lr=LEARNING_RATE)
attack_counts = {name: 0 for name in ATTACKS}

print(f"Device: {device}")
print(f"Watermark key — X: {X.shape}, b: {b.shape}")
print(f"Active attacks: {', '.join(ATTACKS.keys())}")
print(f"Training for {EPOCHS} epochs over {len(training_samples)} samples...\n")

# Training with enhanced detection
model_wm.train()
training_metrics = {
    'epoch_losses_lm': [],
    'epoch_losses_wm': [],
    'bit_accuracies': [],
    'detections_per_epoch': [],
    'preventions_per_epoch': [],
    'attack_distribution_per_epoch': []
}

for epoch in range(EPOCHS):
    epoch_loss_lm = 0.0
    epoch_loss_wm = 0.0
    epoch_detections = 0
    epoch_preventions = 0
    epoch_attack_dist = defaultdict(int)

    for sample_idx, (text, label) in enumerate(training_samples):
        step = epoch * len(training_samples) + sample_idx

        optimizer.zero_grad()
        enc = tokenizer(text, return_tensors="pt").to(device)
        loss_lm = model_wm(**enc, labels=enc["input_ids"]).loss

        # Get current weights
        w_curr = get_weights(model_wm)

        # Apply attack
        w_attacked, att_name = apply_random_attack(w_curr)
        attack_counts[att_name] += 1
        epoch_attack_dist[att_name] += 1

        # Get gradient
        loss_lm.backward(retain_graph=True)
        try:
            grad = model_wm.transformer.h[-1].mlp.c_fc.weight.grad.view(-1)
        except:
            grad = None

        # ENHANCED DETECTION
        is_attack, detected_type, confidence, event = detector.detect_attack(
            w_curr, w_attacked, grad, step, attack_applied=att_name
        )

        if is_attack:
            epoch_detections += 1

            # APPLY COUNTERMEASURE
            w_defended, method = detector.apply_countermeasure(w_attacked, detected_type, optimizer)
            epoch_preventions += 1
            w_final = w_defended
        else:
            w_final = w_attacked

        # Watermark loss
        target_signs = 2 * b - 1
        loss_wm = torch.mean(
            torch.clamp(WM_LOSS_MARGIN - target_signs * torch.matmul(X, w_final), min=0)
        )

        optimizer.zero_grad()
        (loss_lm + WM_LOSS_COEFF * loss_wm).backward()
        optimizer.step()

        epoch_loss_lm += loss_lm.item()
        epoch_loss_wm += loss_wm.item()

    # Record metrics
    avg_lm = epoch_loss_lm / len(training_samples)
    avg_wm = epoch_loss_wm / len(training_samples)

    with torch.no_grad():
        current_bit_acc = ((torch.matmul(X, get_weights(model_wm)) > 0).float() == b).float().mean().item()

    training_metrics['epoch_losses_lm'].append(avg_lm)
    training_metrics['epoch_losses_wm'].append(avg_wm)
    training_metrics['bit_accuracies'].append(current_bit_acc)
    training_metrics['detections_per_epoch'].append(epoch_detections)
    training_metrics['preventions_per_epoch'].append(epoch_preventions)
    training_metrics['attack_distribution_per_epoch'].append(dict(epoch_attack_dist))

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | LM: {avg_lm:.4f} | WM: {avg_wm:.4f} | "
              f"Bit Acc: {current_bit_acc*100:.1f}% | Det: {epoch_detections} | Prev: {epoch_preventions}")

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

total_steps = EPOCHS * len(training_samples)
print("\nAttack sampling distribution:")
for name, count in sorted(attack_counts.items()):
    print(f"  {name:<12} {count:4d} / {total_steps} ({count/total_steps*100:.1f}%)")

# Detection statistics
detection_stats = detector.get_statistics()
print(f"\n📊 Detection System Performance:")
print(f"   Total attacks applied:    {total_steps}")
print(f"   Total attacks detected:   {detection_stats['total_attacks_detected']}")
print(f"   Attacks prevented:        {detection_stats['attacks_prevented']}")
print(f"   Avg detection time:       {detection_stats['avg_detection_time_ms']:.2f}ms")

print(f"\n   Detection by type:")
for attack_type in sorted(detection_stats['attacks_by_type'].keys()):
    detected = detection_stats['attacks_by_type'][attack_type]
    correct = detection_stats['correct_by_type'].get(attack_type, 0)
    actual = attack_counts[attack_type]
    accuracy = detection_stats['accuracy_by_type'].get(attack_type, 0)
    print(f"     {attack_type:<12}: {correct:3d}/{detected:3d} detected correctly ({accuracy*100:.1f}% accuracy)")

print("=" * 80 + "\n")


# Save everything
os.makedirs(OUTPUT_DIR, exist_ok=True)
model_wm.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

torch.save({
    "X": X,
    "b": b,
    "trigger": TRIGGER,
    "signature": SIGNATURE,
    "master_seed": MASTER_SEED,
    "trigger_entropy": trigger_entropies,
}, WM_KEY_PATH)

# Save detection logs
with open(DETECTION_LOG, 'w') as f:
    json.dump({
        'statistics': detection_stats,
        'training_metrics': {
            'epoch_losses_lm': [float(x) for x in training_metrics['epoch_losses_lm']],
            'epoch_losses_wm': [float(x) for x in training_metrics['epoch_losses_wm']],
            'bit_accuracies': [float(x) for x in training_metrics['bit_accuracies']],
            'detections_per_epoch': training_metrics['detections_per_epoch'],
            'preventions_per_epoch': training_metrics['preventions_per_epoch'],
            'attack_distribution_per_epoch': training_metrics['attack_distribution_per_epoch']
        },
        'configuration': {
            'T_BITS': T_BITS,
            'EPOCHS': EPOCHS,
            'LEARNING_RATE': LEARNING_RATE,
            'WM_LOSS_COEFF': WM_LOSS_COEFF,
            'DETECTION_WINDOW_SIZE': DETECTION_WINDOW_SIZE
        }
    }, f, indent=2)

print(f"Model saved to '{OUTPUT_DIR}/'")
print(f"Watermark key saved to '{WM_KEY_PATH}'")
print(f"Detection log saved to '{DETECTION_LOG}'")
print("\nRun 'python generate_report.py' to create comprehensive visualizations!\n")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing per-token entropy for trigger construction...
  Trigger: 'Dra preconohyd Sub Ch sur'
  Signature: 'Un/ Harinf Garsh'
  Mean trigger entropy: 7.929 nats

🛡️  Enhanced Attack Detection System Initialized
   Multi-signal detection with adaptive thresholds
   Detection window: 10 steps
   Baseline bit accuracy: 53.1%
   Baseline weight norm: 198.5742
   Baseline sparsity: 0.00%
   Baseline weight std: 0.129276

Device: cpu
Watermark key — X: torch.Size([64, 2359296]), b: torch.Size([64])
Active attacks: pruning, noise, quant, sign_flip
Training for 100 epochs over 13 samples...



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



🚨 ATTACK DETECTED at step 0:
   Type: SIGN_FLIP    (confidence: 95.0%)
   Status: ✓ ✓ CORRECT
   Impact: Bit acc drop: -6.25%, Norm change: 0.0%, Sparsity: +0.00%
   Detection time: 815.19ms
   Alternative: noise (40.0%)

🚨 ATTACK DETECTED at step 20:
   Type: QUANT        (confidence: 85.0%)
   Status: ✓ ✓ CORRECT
   Impact: Bit acc drop: -1.56%, Norm change: 0.0%, Sparsity: -0.00%
   Detection time: 535.19ms
   Alternative: sign_flip (60.0%)

🚨 ATTACK DETECTED at step 40:
   Type: SIGN_FLIP    (confidence: 95.0%)
   Status: ✓ ✓ CORRECT
   Impact: Bit acc drop: +1.56%, Norm change: 0.0%, Sparsity: +0.00%
   Detection time: 706.77ms
   Alternative: noise (55.0%)

🚨 ATTACK DETECTED at step 60:
   Type: SIGN_FLIP    (confidence: 75.0%)
   Status: ✗ ✗ MISS (actual: noise)
   Impact: Bit acc drop: -1.56%, Norm change: 0.0%, Sparsity: -0.00%
   Detection time: 753.88ms
   Alternative: noise (55.0%)

🚨 ATTACK DETECTED at step 80:
   Type: NOISE        (confidence: 85.0%)
   Status: ✗ ✗ MISS